<a href="https://colab.research.google.com/github/vbrijesh-gmail/JSON_AGENT/blob/main/Pretrained_EfficientNet_(STIR_MRI).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP 1 — Imports
!pip install pydicom
import os
import cv2
import numpy as np
import pydicom
import pandas as pd
import matplotlib.pyplot as plt

from pydicom.pixels import apply_voi_lut

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report


In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#STEP 2 — Preprocessing Function
# IMG_SIZE = 224

# def preprocess_dicom(path):
#     ds = pydicom.dcmread(path)
#     img = apply_voi_lut(ds.pixel_array, ds)

#     if ds.PhotometricInterpretation == "MONOCHROME1":
#         img = np.max(img) - img

#     img = img.astype(np.float32)
#     img = (img - img.min()) / (img.max() - img.min() + 1e-8)

#     img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
#     img = np.expand_dims(img, axis=-1)

#     return img
IMG_SIZE = 224

def preprocess_dicom(path):
    ds = pydicom.dcmread(path)
    img = apply_voi_lut(ds.pixel_array, ds)

    if ds.PhotometricInterpretation == "MONOCHROME1":
        img = np.max(img) - img

    img = img.astype(np.float32)

    # scale to 0–255
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img = img * 255.0

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = np.expand_dims(img, axis=-1)

    return img



In [ ]:
#STEP 3 — Load Slice Files
negative_dir = "/content/drive/MyDrive/spondiloarthritis/sacroilitis/STIR/Negative"
positive_dir = "/content/drive/MyDrive/spondiloarthritis/sacroilitis/STIR/Positive"

import os

def get_dicom_files(folder):
    files = []
    for root, dirs, filenames in os.walk(folder):
        for f in filenames:
            if f.lower().endswith(".dcm"):
                files.append(os.path.join(root, f))
    return files

neg_files = get_dicom_files(negative_dir)
pos_files = get_dicom_files(positive_dir)

print("Negative:", len(neg_files))
print("Positive:", len(pos_files))

Negative: 168
Positive: 120


In [ ]:
# STEP 4 — Build X and y
X = []
y = []

for f in neg_files:
    X.append(preprocess_dicom(f))
    y.append(0)   # Normal

for f in pos_files:
    X.append(preprocess_dicom(f))
    y.append(1)   # Affected

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)


(288, 224, 224, 1) (288,)


In [ ]:
# STEP 5 — Convert 1-Channel → 3-Channel (EfficientNet expects RGB)
X = np.repeat(X, 3, axis=-1)
print(X.shape)


(288, 224, 224, 3)


In [ ]:
# STEP 6 — Train / Validation Split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)

Train: (230, 224, 224, 3) (230,)
Val: (58, 224, 224, 3) (58,)


In [ ]:
# STEP 7 — Class Weights
class_weights = {
    0: 1.0,
    1: len(neg_files) / len(pos_files)
}

print(class_weights)

{0: 1.0, 1: 1.4}


In [ ]:

# STEP 8 — Build Pretrained EfficientNet Modelfrom tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, models
import tensorflow as tf

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False   #    # freeze backbone

model = models.Sequential([
    layers.Input(shape=(224,224,3)),
    layers.Lambda(lambda x: preprocess_input(x)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()




16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,213,668 (16.07 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
# STEP 9 — Train
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=16,
    class_weight=class_weights,
    callbacks=[early_stop]
)

In [ ]:
# NEXT STEP — Fine-Tune EfficientNet
# Unfreeze Top Layers

base_model.trainable = True

# Freeze earlier layers, train only last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

In [ ]:
# Recompile with Lower LR
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
# train again
history_finetune = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=16,
    class_weight=class_weights,
    callbacks=[early_stop]
)

In [ ]:
# STEP 10 — Slice-Level Evaluation


from sklearn.metrics import confusion_matrix, classification_report

y_pred_prob = model.predict(X_val)
y_pred = (y_pred_prob >= 0.4).astype(int)

print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))


2/2 ━━━━━━━━━━━━━━━━━━━━ 29s 15s/step
[[19 15]
 [ 2 22]]
              precision    recall  f1-score   support

           0       0.90      0.56      0.69        34
           1       0.59      0.92      0.72        24

    accuracy                           0.71        58
   macro avg       0.75      0.74      0.71        58
weighted avg       0.78      0.71      0.70        58



In [ ]:
# Next Important Step — Tune Threshold
for t in [0.3, 0.4, 0.45, 0.5]:
    y_pred = (y_pred_prob >= t).astype(int)
    print("\nThreshold:", t)
    print(confusion_matrix(y_val, y_pred))
    print(classification_report(y_val, y_pred))

In [ ]:
#Saving file
model.save("/content/drive/MyDrive/sacroiliitis_efficientnet_final.keras")
print("Model saved successfully!")